### 300.最长递增子序列
给你一个整数数组 nums ，找到其中最长严格递增子序列的长度。

子序列 是由数组派生而来的序列，删除（或不删除）数组中的元素而不改变其余元素的顺序。例如，[3,6,2,7] 是数组 [0,3,1,6,2,2,7] 的子序列。

示例 1：

输入：nums = [10,9,2,5,3,7,101,18]
输出：4

解释：最长递增子序列是 [2,3,7,101]，因此长度为 4。

示例 2：

输入：nums = [0,1,0,3,2,3]
输出：4

示例 3：
输入：nums = [7,7,7,7,7,7,7]
输出：1


#### 1.贪心 + 二分（求最长可想到贪心）
贪心的本质是选择每一阶段的局部最优，从而达到全局最优。

由于输入数组无序，但子序列递增，因此，可以利用贪心算法，选择子序列等长时，尾部元素最小。
1. 维护一个数组 tails, tails[i] 表示长度为 i+1 的递增子序列的末尾元素。
2. tails 数组本身一定是**严格递增**的；
   1. 论证：若 tails[i-1] > tails[i];则tails[i-1]对应的子序列 a ，与 tails[i]对应的子序列 b ，存在 a [i-1] > b[i]，而 b[i-1] < b[i], 即 b[i-1] < a[i], tails[i-1]应记录 b[i-1]，而不是 a[i-1]，矛盾。
3. 遍历数组：
   1. 如果 x 大于 tails 的最后一个元素，则将 x 插入 tails 的末尾，形成一个更长的子序列。
   2. 如果 x 不大于 tails 的最后一个元素，说明不能延长最长长度。但是，x 可能可以替换掉 tails 中某个已有的元素，使得该长度的子序列结尾变得更小。
      1. 查找 tails 中第一个大于 或 等于x的元素， 此位置可以使用二分查找，时间复杂度 O(logn)。
      2. 对于每一个长度 L ，只关心结尾最小的那个子序列。

In [3]:
from typing import List
class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        # 二分查找+贪心
        # tials[i]记录整个数组中，长度为 i + 1的子序列尾部最小值
        tails = [nums[0]] 
        for num in nums:
            i, j = 0, len(tails)  #遍历 当前tails数组

            if num > tails[-1]: #大于最长序列尾，可加入
                tails.append(num)
            else:
                # 二分法查找 tails 中 首个大于 num 的数
                # 为何用二分？ tails必然递增
                while i < j:
                    m = (i + j) // 2 #向下取整
                    if tails[m] < num: # 若要求非严格递增，换为 <=
                        i = m + 1
                    else:
                        j = m # 可能 tails[m]= num
                # 返回的 i 即为 首个大于位置（单调递增）
                tails[i] = num # 替换
                
        res =  len(tails)# res为当前 最长子序列 即tails长度
        return res
    
# nums = [10,9,2,5,3,7,101,18]
nums = [0,1,0,3,2,3]
print(Solution().lengthOfLIS(nums))


4


#### 2.动态规划
1. 状态定义：
   - dp[i] 的值代表 nums 以 nums[i] 结尾的最长子序列长度。
2. 转移方程： 设 `j∈[0,i)`，考虑每轮计算新 dp[i] 时，遍历 `[0,i) `列表区间，做以下判断：
    1. 当 nums[i]>nums[j] 时： nums[i] 可以接在 nums[j] 之后（此题要求严格递增），此情况下最长上升子序列长度为 dp[j]+1 ；
    2. 当 nums[i]<=nums[j] 时： nums[i] 无法接在 nums[j] 之后，此情况上升子序列不成立，跳过。
    3. 上述所有 1. 情况 下计算出的 dp[j]+1 的最大值，为直到 i 的最长上升子序列长度（即 dp[i] ）。实现方式为遍历 j 时，每轮执行 dp[i]=max(dp[i],dp[j]+1)。
    4. 转移方程： `dp[i] = max(dp[i], dp[j] + 1) for j in [0, i)`。
3. 初始状态：
dp[i] 所有元素置 1，含义是每个元素都至少可以单独成为子序列，此时长度都为 1。
返回值：

4. 返回 dp 列表最大值，即可得到全局最长上升子序列长度。

时间复杂度 O(N^2 )：遍历计算 dp 列表需 O(N)，计算每个 dp[i] 需 O(N)。

空间复杂度 O(N)：dp 列表占用线性大小额外空间。

In [ ]:
class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        # dp[i]表示i之前包括i的以nums[i]结尾的最长递增子序列的长度
        # dp[i] = max(dp[j]) + 1，其中 0 <= j < i 且 nums[j] < nums[i]
        # 每一个i，对应的dp[i]（即最长递增子序列）起始大小至少都是1

        if len(nums) <= 1:
            return len(nums)

        dp = [1] * len(nums)
        res = 1

        for i in  range(1, len(nums)): #由于dp[i]初始化为1，且dp[i]要与之前的状态比较，所以从1开始 
            for j in range(0, i):
                if nums[i] > nums[j]:
                    dp[i] = max(dp[i], dp[j] + 1) #取0 ~ i之间最大子串

            res = max(res, dp[i]) #取每个i状态下最长子序列

        return res